# Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from torchsummary import summary
import dask.dataframe as dd
from tokenizers import decoders
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
df1 = dd.read_parquet("hf://datasets/openbmb/Ultra-FineWeb-L3/data/ultrafineweb_en_l3/multi_style/part-00002-f36f5a53-4a77-434b-b9bc-67ed69b93fe2-c000.snappy.parquet")
df2 = dd.read_parquet("hf://datasets/openbmb/Ultra-FineWeb-L3/data/ultrafineweb_en_l3/multi_style/part-00003-f36f5a53-4a77-434b-b9bc-67ed69b93fe2-c000.snappy.parquet")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/8k_tokenizer")

In [ ]:
size_d1 = 85519
size_d2 = 85019
df1 = df1.head(size_d1)
df2 = df2.head(size_d2)

# size_d1 = 85519   2
# size_d2 = 85019   3


tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/8k_tokenizer")

ds = ['' for i in range(size_d1 + size_d2)]
for i in range(size_d1):
  ds[i] = df1.iloc[i]['content']
for i in range(size_d2):
  ds[size_d1+i] = df2.iloc[i]['content']

size_d = size_d1+size_d2


flat_tokens = []
batch_size = 1000

for i in range(0, len(ds), batch_size):
    batch = ds[i : i + batch_size]
    batch_encoded = tokenizer(batch, add_special_tokens=False)['input_ids']

    for seq in batch_encoded:
        flat_tokens.extend(seq)
        flat_tokens.append(3)

    del batch_encoded
    gc.collect()

del ds
gc.collect()

flat_tensor = torch.tensor(flat_tokens,dtype=torch.int16, device=device)

total_token = len(flat_tensor)
context_len = 1024
lleenn = len(flat_tensor)
lleenn = (lleenn // 1024) * 1024

flat_tensor = flat_tensor[:lleenn]
data_tensor = flat_tensor.view(lleenn//1024 , 1024)
del flat_tensor


In [ ]:
data_tensor.shape

torch.Size([94967, 1024])

In [ ]:
train_tensor = data_tensor[:94464]
test_tensor = data_tensor[94464:]

train_tensor = train_tensor.to(device)
test_tensor = test_tensor.to(device)
#del data_tensor

size_train = 94464
size_test = 557

In [ ]:
train_tensor.shape,test_tensor.shape

(torch.Size([94464, 1024]), torch.Size([503, 1024]))

In [ ]:
batch_size = 64
size_t_b = size_train // batch_size
flat_train_x = train_tensor.flatten()
del train_tensor
flat_train_y = torch.cat((flat_train_x,torch.tensor([3],dtype=torch.int16, device=device)),dim=-1)[1:]
flat_train_x.shape,flat_train_y.shape

(torch.Size([96731136]), torch.Size([96731136]))

In [ ]:
train_x = flat_train_x.view([8,8 * 1476,1024]).to(device)
train_y = flat_train_y.view([8,8 * 1476,1024]).to(device)
del flat_train_x
del flat_train_y

In [ ]:
flat_test_x = test_tensor.flatten()[:(500 * 1024)]
del test_tensor
flat_test_y = torch.cat((flat_test_x, torch.tensor([3], dtype=torch.int16, device=device)),dim=-1)[1:]
flat_test_x.shape,flat_test_y.shape

(torch.Size([512000]), torch.Size([512000]))

In [ ]:
test_x = flat_test_x.view([4,125,1024]).to(device)
test_y = flat_test_y.view([4,125,1024]).to(device)
del flat_test_x
del flat_test_y

In [ ]:
gc.collect()

48

# Dense Transformer

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from torchsummary import summary
import torch.nn.functional as F

head_size = 64
hdim = 256
max_context_len = 1024
device = 'cuda' if torch.cuda.is_available() else 'cpu'


context_pos = torch.arange(max_context_len).float()
inv_freq = 1.0 / (10000 ** (torch.arange(0, head_size, 2).float() / head_size))
freq = torch.outer(context_pos,inv_freq)
freq_rot = torch.cat((freq, freq), dim=-1)

cached_sin = torch.sin(freq_rot).to(device)
cached_cos = torch.cos(freq_rot).to(device)


class SwiGLU(nn.Module):
  def __init__(self,in_size = 256 ,out_size = 896):
    super().__init__()
    self.swiwei = nn.Linear(in_size,out_size,bias=False)
    self.outwei = nn.Linear(in_size,out_size,bias=False)
    self.finalwei = nn.Linear(out_size,in_size,bias=False)
    self.silu = nn.SiLU()
  def forward(self,x):
    return self.finalwei((self.silu(self.swiwei(x))) * self.outwei(x))


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_RoPE(sin, cos, q, k):
    q_out = (q * cos) + (rotate_half(q) * sin)
    k_out = (k * cos) + (rotate_half(k) * sin)
    return q_out, k_out

class GQAN_1(nn.Module):
  def __init__(self,head_size=64,hdim=256,n = 4):
    super().__init__()
    self.key_mat = nn.Linear(hdim,head_size,bias=False)
    self.query_mat = nn.Linear(hdim,head_size * n,bias=False)
    self.value_mat = nn.Linear(hdim,head_size,bias=False)
    self.n = n
    self.head_size = head_size

  def forward(self,x):
    B,T,_ = x.shape                                                     # 1,1024,h_dim
    cach_cos = cached_cos[:T]
    cach_sin = cached_sin[:T]
    k = self.key_mat(x).view([B,T,1,self.head_size]).transpose(1,2)     # B,1,1024,64
    v = self.value_mat(x).view([B,T,1,self.head_size]).transpose(1,2)
    q = self.query_mat(x).view([B,T,self.n,self.head_size]).transpose(1,2)

    q,k = apply_RoPE(cach_sin,cach_cos,q,k)

    out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=0.1 if self.training else 0.0,
            is_causal=True,
            enable_gqa=True
    )

    out = out.transpose(1, 2).contiguous().view(B, T, self.n * self.head_size)
    return out

class GQA(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = GQAN_1(head_size=64,hdim=192,n=3)
        self.layer2 = GQAN_1(head_size=64,hdim=192,n=3)
        self.down_weight1 = nn.Linear(256,192,bias=False)
        self.down_weight2 = nn.Linear(256,192,bias=False)
        self.upscale = nn.Linear(192 + 192,256,bias=False)
        self.wei_g1 = nn.Linear(256,192,bias=False)
        self.wei_g2 = nn.Linear(256,192,bias=False)
        self.rms_b = nn.RMSNorm(256)
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.wei_g1.weight)
        nn.init.zeros_(self.wei_g2.weight)
    def forward(self,x):
        norm_x = self.rms_b(x)
        b1 =  self.layer1(self.down_weight1(norm_x))
        b2 =  self.layer2(self.down_weight2(norm_x))
        h1 = self.sigmoid(self.wei_g1(norm_x)) * b1
        h2 = self.sigmoid(self.wei_g2(norm_x)) * b2
        b = torch.cat((h1,h2),dim=-1)
        return self.upscale(b)

class total_attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.gqa = GQA()
        self.swiglu = SwiGLU(in_size = 256 ,out_size = 1024) #896
        self.rms = nn.RMSNorm(256)
    def forward(self,x):
        attn = self.gqa(x)
        h = attn + x
        norm_x = self.rms(h)
        ff = self.swiglu(norm_x)
        return attn + ff

In [ ]:
class DenseTrans(torch.nn.Module):
    def __init__(self,hdim=256,vocab=8192,layers=11):
        super().__init__()
        self.emb = nn.Embedding(vocab,hdim)
        self.attention_block = nn.ModuleList([total_attention() for _ in range(layers)])
        self.rms = nn.RMSNorm(hdim)
        self.layers = layers
    def forward(self,x):
        x = self.emb(x)
        for i in range(self.layers):
            x = x + self.attention_block[i](x)
        x = self.rms(x)
        return (x @ self.emb.weight.transpose(1,0))

In [ ]:
!pip install torchinfo
from torchinfo import summary
model = DenseTrans().to(device)
summary(model,input_data=torch.randint(0,8192,[1,1024],device=device))

Layer (type:depth-idx)                   Output Shape              Param #
DenseTrans                               [1, 1024, 8192]           --
├─Embedding: 1-1                         [1, 1024, 256]            2,097,152
├─ModuleList: 1-2                        --                        --
│    └─total_attention: 2-1              [1, 1024, 256]            --
│    │    └─GQA: 3-1                     [1, 1024, 256]            418,048
│    │    └─RMSNorm: 3-2                 [1, 1024, 256]            256
│    │    └─SwiGLU: 3-3                  [1, 1024, 256]            786,432
│    └─total_attention: 2-2              [1, 1024, 256]            --
│    │    └─GQA: 3-4                     [1, 1024, 256]            418,048
│    │    └─RMSNorm: 3-5                 [1, 1024, 256]            256
│    │    └─SwiGLU: 3-6                  [1, 1024, 256]            786,432
│    └─total_attention: 2-3              [1, 1024, 256]            --
│    │    └─GQA: 3-7                     [1, 1024, 256] 

In [ ]:
model = torch.compile(loaded_model).to(device)

In [ ]:
decay_params = []
no_decay_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.dim() >= 2:
        decay_params.append(param)
    else:
        no_decay_params.append(param)

optim_groups = [
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': no_decay_params, 'weight_decay': 0.0}
]

In [ ]:
from transformers import get_cosine_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

l_r = 8e-4
batch_size = 64
optimizer = torch.optim.AdamW(
    optim_groups,
    lr = l_r,
    betas=(0.9,0.95),
    eps=1e-8
)

total_steps = 4500
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=200,
    num_training_steps=total_steps
)

/tmp/ipykernel_1111/3489697487.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.13/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [ ]:
@torch.no_grad()
def estimate_val_loss(model, val_x, val_y, max_batches=164):
    model.eval()

    total_loss = 0.0
    num_steps = min(val_x.size(1), max_batches)

    for i in range(num_steps):
        x = val_x[:, i, :]
        y = val_y[:, i, :]
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(x)
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, 8192),
                y.reshape(-1)
            )

        total_loss += loss.item()

    model.train()
    return total_loss / num_steps

In [ ]:
model.train()

print_num = 100
global_step = 0
eval_interval = 500

epoch = 1

micro_batch_size = 8
accumulation_steps = 8



for e in range(epoch):

    for i in range(8 * (800), 8 * (1467)): # next 800 --> 1467

        # Take only a micro-batch
        x = train_x[:, i, :].to(torch.long)
        y = train_y[:, i, :].to(torch.long)

        with torch.autocast(device_type='cuda', dtype=torch.float16):

            logits = model(x)

            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, 8192),
                y.reshape(-1)
            )

            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if ((i + 1) % accumulation_steps == 0):

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()

            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            if (global_step % print_num) == 0:
                print(
                    f"Epoch {e} | "
                    f"Step {global_step} | "
                    f"Loss: {loss.item() * accumulation_steps:.4f}"
                )


Epoch 0 | Step 100 | Loss: 4.1678
Epoch 0 | Step 200 | Loss: 3.9963
Epoch 0 | Step 300 | Loss: 3.9567
Epoch 0 | Step 400 | Loss: 3.9278
Epoch 0 | Step 500 | Loss: 3.8395
Epoch 0 | Step 600 | Loss: 3.8292


In [ ]:
torch.cuda.empty_cache()

In [ ]:

torch.save(
    (model._orig_mod if hasattr(model, "_orig_mod") else model).state_dict(),
    "/content/drive/MyDrive/dense_trans_15M4.pth"
)

In [ ]:
# 1. Load the raw state_dict
raw_state_dict = torch.load("/content/drive/MyDrive/dense_trans_15M4.pth", weights_only=True,map_location=torch.device('cpu'))

# If you saved a full checkpoint dict, extract it first:
# raw_state_dict = raw_state_dict['model_state_dict']

# 2. Strip the "_orig_mod." prefix from every key
clean_state_dict = {
    key.replace("_orig_mod.", ""): value
    for key, value in raw_state_dict.items()
}

# 3. Instantiate and load
loaded_model = DenseTrans().to(device)
loaded_model.load_state_dict(clean_state_dict)
loaded_model.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
val_loss = estimate_val_loss(
  loaded_model,
  test_x.to(torch.long),
  test_y.to(torch.long),
)
val_loss  #4.59 for 1 epoch 4.08 for 2 3.95 for 3 ) on 0,1            for 1, for 2, for 3 ) on 2,3

3.7815656280517578

In [ ]:
model(torch.randint(0,8192,[1,1024],device=device))

tensor([[[-14.4870, -11.9614, -13.3623,  ...,  -5.0449,  -3.2269,  -5.3418],
         [-15.3645, -12.7301, -12.1467,  ...,   1.5590,  -1.5881,  -5.6906],
         [-15.5369, -14.3557, -10.9735,  ...,  -6.6691,  -4.4692,  -3.2361],
         ...,
         [-11.2935,  -9.4924, -17.7507,  ...,  -2.0084,  -4.9023,  -3.7978],
         [-16.5654,  -9.0045, -11.6818,  ...,  -3.0798,  -1.9718,  -5.3163],
         [-15.7569, -11.0405, -14.5147,  ...,  -4.1939,  -1.0831,  -4.7391]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 150,
    temperature: float = 0.8,
    top_k: int = 40,
    stop_token_id: int = 3
):
    model.eval()

    input_ids = tokenizer.encode(prompt, add_special_tokens=False)
    x = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0) #
    print(f"\n--- PROMPT: \"{prompt}\" ---")

    for _ in range(max_new_tokens):
        x_cond = x[:, -1024:] if x.size(1) > 1024 else x

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(x_cond) # Shape: [1, T, 8192]

        next_token_logits = logits[:, -1, :] # Shape: [1, 8192]


        next_token_logits = next_token_logits / max(temperature, 1e-5)

        if top_k is not None:
            v, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
            next_token_logits[next_token_logits < v[:, [-1]]] = -float('Inf')

        probs = F.softmax(next_token_logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        if next_token.item() == stop_token_id:
            break

        x = torch.cat((x, next_token), dim=1)

    generated_ids = x.squeeze(0).tolist()
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return generated_text

In [ ]:
# Test 1: Narrative story prompt
print(generate_text(
    loaded_model,
    tokenizer,
    prompt="Once upon a time, in a small village,",
    max_new_tokens=100,
    temperature=0.7,
    top_k=30
))

# Test 2: Fact/information prompt
print(generate_text(
    loaded_model,
    tokenizer,
    prompt="The history of science shows that",
    max_new_tokens=100,
    temperature=0.8,
    top_k=40
))

# Test 3: Technical/logic prompt
print(generate_text(
    loaded_model,
    tokenizer,
    prompt="In order to build a computer,",
    max_new_tokens=100,
    temperature=0.7,
    top_k=30
))


--- PROMPT: "Once upon a time, in a small village," ---


/tmp/ipykernel_7688/3462634743.py:23: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.autocast(device_type='cuda', dtype=torch.float16):


Once upon a time, in a small village, are used to create a sense of self-awareness. A new muzzle, also known as the "green", is a cousin that can be used as a bag to sink and sink, which is a major human being. The term "green" is derived from a muzzle, meaning "guchi" or "guchi" in the crown.

The term "guchi" refers to a m

--- PROMPT: "The history of science shows that" ---
The history of science shows that the term "law" remains evident across its medical societies. The text further includes key concepts and findings, including the concept of the BBC, which aims to illustrate how the term is used in medical settings, the term "law rhym", and the idea of a "laws" meaning "law." The text also references the concept of "law rhym" and "law tatto" in a study used to describe the idea of

--- PROMPT: "In order to build a computer," ---
In order to build a computer, the system must navigate through the device and rewrite the computer to the computer.

To help users meet the standards of t

In [ ]:
generate_text(
    loaded_model,
    tokenizer,
    prompt="Hey How are you",
    max_new_tokens=108,
    temperature=0.7,
    top_k=30
)


--- PROMPT: "Hey How are you" ---


/tmp/ipykernel_7688/3462634743.py:23: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.autocast(device_type='cuda', dtype=torch.float16):


'Hey How are you for your presence for your cow’s environment, your value, and the ability to perceive your cow’s inner world.\n\nThe primary opportunity to help you become meaningful. Did you know you’ll be for your inner world?  \nAmong the most effective steps to ensure your cow’s life are active, and you’ll have a strong sense of personal connection. You’re in the face of a state of social connection and connection.\n\nThe Snake: Set your'